# ComfyUI + Z-Image Turbo - 2816x1584 生成

## 手順
1. ランタイム → GPU T4以上を選択
2. すべてのセルを実行
3. HuggingFace tokenを求められたら入力

In [ ]:
# GPU確認
import torch
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# HuggingFace token入力
print("HuggingFace tokenを入力してください")
print("(https://huggingface.co/settings/tokens で取得可能)")
HF_TOKEN = input("token: ")

In [ ]:
# ComfyUIインストール（存在しない場合のみ）
import os
import subprocess

COMFYUI_PATH = "/content/ComfyUI"

if not os.path.exists(COMFYUI_PATH):
    print("ComfyUIフォルダが見つからないため、インストールを開始...")
    !git clone https://github.com/comfyanonymous/ComfyUI.git {COMFYUI_PATH}
    %cd {COMFYUI_PATH}
    !pip install -r requirements.txt -q
    print("ComfyUIインストール完了")
else:
    print(f"ComfyUIは既に存在します: {COMFYUI_PATH}")
    %cd {COMFYUI_PATH}

# フォルダ確認
print(f"\n/content 内容:")
for item in os.listdir("/content"):
    print(f"  - {item}")

In [ ]:
# Z-Image Turboモデルをダウンロード
import os
from huggingface_hub import login, hf_hub_download

COMFYUI_PATH = "/content/ComfyUI"
CHECKPOINT_PATH = f"{COMFYUI_PATH}/models/checkpoints"
os.makedirs(CHECKPOINT_PATH, exist_ok=True)

login(token=HF_TOKEN)

# 既にモデルがあるか確認
model_file = f"{CHECKPOINT_PATH}/z-image-turbo-fp8-aio.safetensors"
if os.path.exists(model_file):
    size = os.path.getsize(model_file)
    print(f"モデルは既に存在します: {size / (1024**3):.2f} GB")
else:
    print("Downloading Z-Image Turbo model (FP8, ~10GB)...")
    hf_hub_download(
        repo_id="SeeSee21/Z-Image-Turbo-AIO",
        filename="z-image-turbo-fp8-aio.safetensors",
        local_dir=CHECKPOINT_PATH
    )
    size = os.path.getsize(model_file)
    print(f"Download complete! File size: {size / (1024**3):.2f} GB")

In [ ]:
# ComfyUI起動（ログ出力付き）
import subprocess
import threading
import time
import os

COMFYUI_PATH = "/content/ComfyUI"

# パス確認
if not os.path.exists(COMFYUI_PATH):
    print(f"ERROR: {COMFYUI_PATH} が存在しません！")
    print("前のセル（ComfyUIインストール）を実行してください")
    raise FileNotFoundError(f"{COMFYUI_PATH} not found")

if not os.path.exists(f"{COMFYUI_PATH}/main.py"):
    print(f"ERROR: {COMFYUI_PATH}/main.py が存在しません！")
    raise FileNotFoundError(f"{COMFYUI_PATH}/main.py not found")

print(f"ComfyUIパス確認OK: {COMFYUI_PATH}")

# 既存のComfyUIプロセスを終了
os.system("pkill -f 'python main.py' 2>/dev/null || true")
time.sleep(2)

def run_comfyui():
    with open("/content/comfyui_output.log", "w") as f:
        proc = subprocess.Popen(
            ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
            cwd=COMFYUI_PATH,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )
        for line in proc.stdout:
            f.write(line)
            f.flush()

t = threading.Thread(target=run_comfyui, daemon=True)
t.start()

print(f"ComfyUI starting from {COMFYUI_PATH}...")
print("最初の起動には2-3分かかる場合があります...")
time.sleep(120)
print("Initial wait done. Checking status...")

In [ ]:
# ComfyUI接続確認とログ確認
import requests
import time
import os

# ログを表示
print("=== ComfyUI ログ ===")
try:
    with open("/content/comfyui_output.log", "r") as f:
        lines = f.readlines()
        for line in lines:
            print(line.rstrip())
except Exception as e:
    print(f"ログファイル読み取りエラー: {e}")

print("\n=== 接続確認 ===")
for i in range(120):
    try:
        r = requests.get("http://127.0.0.1:8188/system_stats", timeout=5)
        print(f"✓ ComfyUI connected! ({i+1}秒)")
        print(f"  Response: {r.json()}")
        break
    except Exception as e:
        if i % 20 == 0:
            print(f"Waiting... {i+1}/120 - {str(e)[:50]}")
        time.sleep(1)
else:
    print("✗ Connection failed after 120 seconds")
    print("\n=== ログの最後 ===")
    try:
        with open("/content/comfyui_output.log", "r") as f:
            lines = f.readlines()
            for line in lines[-50:]:
                print(line.rstrip())
    except:
        pass

In [ ]:
# VRAM使用量確認
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"Cached: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
    
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    used = torch.cuda.memory_allocated() / 1024**3
    free = total - used
    print(f"Free VRAM: ~{free:.1f} GB")
    
    if free < 8:
        print("\n⚠️ VRAM不足の可能性！ 解像度を落とすか、モデルを変える必要があるかも")

In [ ]:
# 利用可能なモデル一覧取得
import requests

models = requests.get("http://127.0.0.1:8188/api/models").json()
print("利用可能なcheckpoint:")
for m in models:
    if isinstance(m, dict):
        if "checkpoints" in m.get("model_type", ""):
            print(f"  - {m.get('name', 'unknown')}")
    else:
        print(f"  - {m}")

# プロンプト設定
※ ここを変更して生成内容を変更

In [ ]:
# プロンプト設定（ここに生成したい内容を書く）

POSITIVE_PROMPT = """
東京夜景、ネオンライト、赤い傘を持った男性、雨上がり、スーツビジネススーツ、都市の夜景
"""

NEGATIVE_PROMPT = """
blur, low quality, distorted, watermark, cartoon, anime, ugly, deformed
"""

# 生成設定
WIDTH = 2816
HEIGHT = 1584
SEED = 42
STEPS = 8

# プロンプトからファイル名を生成
def generate_filename(prompt, seed):
    import re
    words = re.findall(r'[\w\u3040-\u309f\u30a0-\u30ff\u4e00-\u9faf]+', prompt)
    keywords = words[:5] if len(words) >= 5 else words
    filename = "_".join(keywords)
    filename = filename.lower().replace(" ", "_")
    return f"stockphoto_{filename}_{seed:05d}"

FILENAME_PREFIX = generate_filename(POSITIVE_PROMPT, SEED)

print(f"Positive: {POSITIVE_PROMPT.strip()}")
print(f"Negative: {NEGATIVE_PROMPT.strip()}")
print(f"Size: {WIDTH}x{HEIGHT}")
print(f"Filename prefix: {FILENAME_PREFIX}")

In [ ]:
# 保存先パス設定
print("画像の保存先パスを入力してください")
print("(例: /content/stockphoto/ または /content/)")
SAVE_PATH = input("保存先パス: ")

# パスが存在しない場合は作成
import os
os.makedirs(SAVE_PATH, exist_ok=True)
print(f"保存先: {SAVE_PATH}")

# 繰り返し生成ループ
※ このセルを実行すると、停止するまで繰り返し生成します
※ プロンプトやシードを変更したい場合は、セルを停止→設定変更→再実行

In [ ]:
# 繰り返し画像生成（停止するまで実行）
import requests
import time
import glob
import os
import shutil
from PIL import Image
from google.colab import files

def generate_filename(prompt, seed):
    import re
    words = re.findall(r'[\w\u3040-\u309f\u30a0-\u30ff\u4e00-\u9faf]+', prompt)
    keywords = words[:5] if len(words) >= 5 else words
    filename = "_".join(keywords)
    filename = filename.lower().replace(" ", "_")
    return f"stockphoto_{filename}_{seed:05d}"

def png_to_jpg(png_path):
    jpg_path = png_path.replace(".png", ".jpg")
    img = Image.open(png_path)
    
    # 解像度強制リサイズ（2816x1584）
    if img.size != (WIDTH, HEIGHT):
        print(f"  元サイズ: {img.size[0]}x{img.size[1]} → 強制リサイズ: {WIDTH}x{HEIGHT}")
        img = img.resize((WIDTH, HEIGHT), Image.LANCZOS)
    
    if img.mode in ('RGBA', 'P'):
        img = img.convert('RGB')
    img.save(jpg_path, 'JPEG', quality=95, optimize=True)
    
    # 保存後にサイズ確認
    verify_img = Image.open(jpg_path)
    print(f"  保存後サイズ確認: {verify_img.size[0]}x{verify_img.size[1]}")
    
    return jpg_path

def save_and_download(jpg_path, save_path):
    filename = os.path.basename(jpg_path)
    dest_path = os.path.join(save_path, filename)
    shutil.copy(jpg_path, dest_path)
    print(f"保存完了: {dest_path}")
    files.download(jpg_path)

count = 0
print("=" * 50)
print("繰り返し生成開始")
print("停止するには ■ ボタン or Runtime → 中断")
print("=" * 50)

while True:
    count += 1
    current_seed = SEED + count - 1
    current_filename = generate_filename(POSITIVE_PROMPT, current_seed)
    
    print(f"\n{'='*50}")
    print(f"生成 #{count} | Seed: {current_seed} | Filename: {current_filename}")
    print(f"{'='*50}")
    
    # ワークフロー作成
    workflow = {
        "1": {"inputs": {"ckpt_name": "z-image-turbo-fp8-aio.safetensors"}, "class_type": "CheckpointLoaderSimple"},
        "2": {"inputs": {"text": POSITIVE_PROMPT.strip(), "clip": ["1", 1]}, "class_type": "CLIPTextEncode"},
        "3": {"inputs": {"text": NEGATIVE_PROMPT.strip(), "clip": ["1", 1]}, "class_type": "CLIPTextEncode"},
        "4": {"inputs": {"width": WIDTH, "height": HEIGHT, "batch_size": 1}, "class_type": "EmptyLatentImage"},
        "5": {"inputs": {"model": ["1", 0], "seed": current_seed, "steps": STEPS, "cfg": 1.0, "sampler_name": "euler", "scheduler": "normal", "positive": ["2", 0], "negative": ["3", 0], "latent_image": ["4", 0], "denoise": 1.0}, "class_type": "KSampler"},
        "6": {"inputs": {"samples": ["5", 0], "vae": ["1", 2]}, "class_type": "VAEDecode"},
        "7": {"inputs": {"images": ["6", 0], "filename_prefix": current_filename}, "class_type": "SaveImage"}
    }
    
    # 画像生成
    print("画像生成中...")
    result = requests.post("http://127.0.0.1:8188/prompt", json={"prompt": workflow}).json()
    
    if "prompt_id" not in result:
        print(f"Error: {result}")
        continue
    
    prompt_id = result['prompt_id']
    
    # 生成完了まで待機
    for i in range(1200):
        time.sleep(1)
        hist = requests.get(f"http://127.0.0.1:8188/history/{prompt_id}").json()
        if prompt_id in hist and hist[prompt_id]['status']['completed']:
            print("生成完了!")
            break
        if i % 60 == 0:
            print(f"  待機中... {i//60}分")
    else:
        print("タイムアウト")
        continue
    
    # PNG → JPG変換（強制リサイズ付き）
    print("JPG変換中（2816x1584強制リサイズ）...")
    png_files = glob.glob(f"/content/ComfyUI/output/{current_filename}_*.png")
    if not png_files:
        print("PNGファイルが見つかりません")
        continue
    
    latest_png = max(png_files, key=os.path.getmtime)
    jpg_path = png_to_jpg(latest_png)
    print(f"JPG変換完了: {os.path.basename(jpg_path)}")
    
    # 保存・ダウンロード
    print("保存・ダウンロード中...")
    save_and_download(jpg_path, SAVE_PATH)
    
    print(f"\n✅ #{count} 完了!")